# ORA - WGCNA Modules (Clinical Outcome)

Runs over-representation analysis (`gseapy.enrich`) for a WGCNA module's hub genes and complete gene set against 5 local MSigDB GMTs. Set `MODULE` below to `"turquoise"` or `"blue"` and run the notebook.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import gseapy as gp
from pathlib import Path

In [ ]:
# config - change MODULE to "turquoise" or "blue" and re-run
MODULE = "blue"

BASE_DIR = Path(f"{ACC_DATA_ROOT}/bulk_rna_data/outputs/images/clinical_outcome/wgcna")
GMT_DIR = Path(f"{ACC_DATA_ROOT}/reference/msigdb_gmt")
OUT_DIR = BASE_DIR / "ora" / MODULE

gene_sets = ["MSigDB_C5_BP.gmt", "MSigDB_C5_CC.gmt", "MSigDB_C5_MF.gmt", "MSigDB_C2_Reactome.gmt", "MSigDB_Hallmark.gmt"]
cutoff = 0.05

In [ ]:
# background universe: genes retained after WGCNA expression filtering
background_genes = pd.read_csv(BASE_DIR / "genes_after_filtering_in_wgcna_for_background_universe_ORA.txt", header=None)[0].to_list()

In [ ]:
# complete module gene set - all genes assigned to MODULE
gene_info = pd.read_csv(BASE_DIR / "06_gene_info_complete.csv")
complete_set = gene_info.loc[gene_info["Module"] == MODULE, "Gene"].to_list()

In [ ]:
# hub genes - top-connectivity genes for MODULE (Cytoscape node table)
hub_genes = pd.read_csv(BASE_DIR / f"13_cytoscape_{MODULE}_nodes.csv")["Gene"].to_list()

In [ ]:
print(f"MODULE: {MODULE}")
print(f"background genes: {len(background_genes)}")
print(f"complete module gene set: {len(complete_set)}")
print(f"hub genes: {len(hub_genes)}")

In [ ]:
# running ORA for both hub genes and the complete module gene set, across all gene_sets
gene_lists = {"hub": hub_genes, "comp_set": complete_set}

for label, genes in gene_lists.items():
    outdir = OUT_DIR / label
    outdir.mkdir(parents=True, exist_ok=True)
    for i in gene_sets:
        try:
            enr = gp.enrich(
                gene_list=genes,
                gene_sets=str(GMT_DIR / i),
                background=background_genes,
                outdir=f"{ACC_DATA_ROOT}/tmp",
                cutoff=cutoff,
            )
            res_sorted = enr.results.sort_values(["Adjusted P-value", "P-value"])
            res_sorted.to_csv(outdir / f"{i}.csv", index=False)
        except (ValueError):
            pass